In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython())
print(f"IN_COLAB={IN_COLAB}")


### Environment detection
This first cell sets `IN_COLAB` so later installation and path logic can branch cleanly between Colab and local execution.

### Install dependencies
This cell installs the right dependency set for the current environment so the same notebook can run in Colab or locally.

In [ ]:
import subprocess
import sys
from pathlib import Path

req_candidates = [Path('requirements_colab.txt'), Path('../requirements_colab.txt')] if IN_COLAB else [Path('requirements.txt'), Path('../requirements.txt')]
req_path = next((p for p in req_candidates if p.exists()), None)
if req_path is None:
    print('Could not find requirements file from current working directory.')
else:
    print(f'Installing from: {req_path.resolve()}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_path)], check=False)


### Mount Drive and configure paths
This sets a stable workspace path and adds the project root to `sys.path` so package imports work consistently.

In [ ]:
import sys
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/piano_model')
    BASE_PATH.mkdir(parents=True, exist_ok=True)
    PROJECT_ROOT = BASE_PATH / 'piano_midi_model'
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
    BASE_PATH = PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
if IN_COLAB and not PROJECT_ROOT.exists():
    print('Upload/clone this repository to the PROJECT_ROOT path above.')


### Verify MAESTRO dataset
Before training a tokenizer, we validate dataset availability and report basic dataset size statistics.

In [ ]:
from pathlib import Path

from config import DataConfig

data_cfg = DataConfig()
if IN_COLAB:
    data_cfg.maestro_path = str(BASE_PATH / 'maestro-v3.0.0')
    data_cfg.tokenizer_path = str(BASE_PATH / 'tokenizer.json')
    data_cfg.processed_path = str(BASE_PATH / 'processed')

maestro_root = Path(data_cfg.maestro_path)
midi_files = sorted(list(maestro_root.rglob('*.mid')) + list(maestro_root.rglob('*.midi'))) if maestro_root.exists() else []
total_bytes = sum(p.stat().st_size for p in midi_files) if midi_files else 0

print(f'MAESTRO path: {maestro_root}')
print(f'File count: {len(midi_files)}')
print(f'Total size: {total_bytes / (1024 ** 3):.2f} GB')
if not maestro_root.exists():
    print('Dataset missing. Download: https://magenta.tensorflow.org/datasets/maestro')


### Initialize and train tokenizer
We use REMI + BPE to build a compact event vocabulary suitable for sequence modeling and generation.

In [ ]:
from pathlib import Path

from data.tokenizer import PianoTokenizer

if len(midi_files) == 0:
    raise RuntimeError('No MIDI files found. Fix data_cfg.maestro_path before proceeding.')

tokenizer_path = Path(data_cfg.tokenizer_path)
if tokenizer_path.exists():
    tokenizer = PianoTokenizer.load(str(tokenizer_path))
    print(f'Loaded tokenizer: {tokenizer_path}')
else:
    tokenizer = PianoTokenizer()
    tokenizer.train(midi_paths=midi_files, vocab_size=data_cfg.vocab_size)
    tokenizer.save(str(tokenizer_path))
    print(f'Trained and saved tokenizer: {tokenizer_path}')

print(f'Vocab size: {tokenizer.vocab_size}')


### Verify tokenizer roundtrip on random samples
This is a critical timing-preservation check: we encode/decode random files and compare original vs reconstructed piano rolls.

In [ ]:
import random
import tempfile
from pathlib import Path

from utils.midi_utils import visualize_pianoroll

sample_files = random.sample(midi_files, k=min(5, len(midi_files)))
for midi_path in sample_files:
    ok = tokenizer.verify_roundtrip(midi_path)
    print(f'{midi_path.name}: {"PASS" if ok else "FAIL"}')
    with tempfile.TemporaryDirectory() as tmp_dir:
        recon_path = Path(tmp_dir) / f'{midi_path.stem}_roundtrip.mid'
        tokens = tokenizer.encode(midi_path)
        tokenizer.decode(tokens, recon_path)
        visualize_pianoroll(midi_path, title=f'Original - {midi_path.name}')
        visualize_pianoroll(recon_path, title=f'Roundtrip - {midi_path.name}')


### Run full MAESTRO preprocessing
This tokenizes the full corpus, filters short pieces, writes token arrays, and builds a manifest for dataset loading.

In [ ]:
from data.preprocess import preprocess_maestro

summary = preprocess_maestro(data_cfg)
summary


### Inspect processed dataset statistics
We plot tokenized piece lengths and vocabulary usage to catch distribution issues early before training.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

manifest_path = Path(data_cfg.processed_path) / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
lengths = np.asarray([m['length'] for m in manifest], dtype=np.int64)

token_counts = np.zeros(tokenizer.vocab_size, dtype=np.int64)
for item in tqdm(manifest, desc='Token frequency'):
    arr = np.load(item['tokens_path'])
    vals, cnts = np.unique(arr, return_counts=True)
    mask = (vals >= 0) & (vals < tokenizer.vocab_size)
    token_counts[vals[mask]] += cnts[mask]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(lengths, bins=40, color='steelblue')
axes[0].set_title('Piece Length Distribution')
axes[0].set_xlabel('Tokens')
axes[0].set_ylabel('Pieces')
axes[1].plot(token_counts, color='darkorange')
axes[1].set_title('Vocabulary Distribution')
axes[1].set_xlabel('Token ID')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print(f'Pieces: {len(lengths)} | mean={lengths.mean():.1f} | min={lengths.min()} | max={lengths.max()}')


### Create DataLoaders and inspect one batch
This confirms seed/continuation tensor shapes and that split-by-piece loading works before model training.

In [ ]:
from config import TrainConfig
from data.dataset import create_dataloaders

train_cfg = TrainConfig(batch_size=8, grad_accumulation_steps=4, max_epochs=1, device='auto')
if IN_COLAB:
    train_cfg.checkpoint_dir = str(BASE_PATH / 'checkpoints_tmp')

train_loader, val_loader, test_loader = create_dataloaders(data_cfg, train_cfg)
seed_batch, continuation_batch = next(iter(train_loader))
print('seed batch:', tuple(seed_batch.shape), seed_batch.dtype)
print('continuation batch:', tuple(continuation_batch.shape), continuation_batch.dtype)


### Visualize one seed/continuation pair
We decode a sampled pair and compare piano rolls to verify the training pair segmentation is musically sensible.

In [ ]:
import tempfile
from pathlib import Path

from utils.midi_utils import compare_pianorolls

seed_tokens, continuation_tokens = train_loader.dataset[0]
seed_list = seed_tokens.tolist()
continuation_list = continuation_tokens.tolist()

with tempfile.TemporaryDirectory() as tmp_dir:
    seed_mid = Path(tmp_dir) / 'seed.mid'
    continuation_mid = Path(tmp_dir) / 'continuation.mid'
    tokenizer.decode(seed_list, seed_mid)
    tokenizer.decode(continuation_list, continuation_mid)
    compare_pianorolls(seed_mid, continuation_mid)
